In [1]:
from dotenv import load_dotenv

load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"


In [2]:
# Helper functions
# Ensure the add_user_messsage and add_assistant_message can handle mutiple message blocks
from anthropic.types import Message

def add_user_message(messages, message):
    user_message = {"role": "user",
                     "content": message.content if isinstance(message, Message) else message
                     }
    messages.append(user_message)

def add_assistant_message(messages, message):
    assistant_message = {"role": "assistant",
                          "content": message.content if isinstance(message, Message) else message
                          }
    messages.append(assistant_message)

# Ensure chat can receive a list of tools. Also, return the full generated message, not just text
def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    if tools:
        params["tools"] = tools
    
    message = client.messages.create(**params)
    return message


def text_from_message(message):
    """Extract all text from text blocks from a message"""
    return "\n".join(
        [block.text for block in message.content if block.type == 'text']
    )

In [3]:
# Tools and Schemas

from datetime import datetime, timedelta


def tool_add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def tool_set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


schema_add_duration_to_datetime = {
    "name": "tool_add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

schema_set_reminder = {
    "name": "tool_set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

schema_batch_tool = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

from anthropic.types import ToolParam

def tool_get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format.strip():
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

schema_get_current_datetime = ToolParam({
  "name": "tool_get_current_datetime",
  "description": "Returns the current date and time formatted according to the specified format string. Use this tool whenever you need to know the current date, time, or both.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A Python strftime-compatible format string that controls how the datetime is returned. Defaults to '%Y-%m-%d %H:%M:%S' (e.g. '2025-01-30 14:35:00'). Common directives: %Y=4-digit year, %m=month, %d=day, %H=hour (24h), %M=minute, %S=second. Must not be empty or whitespace only.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
}
)

In [4]:

messages = []
add_user_message(messages, "What's the current time in HH:MM:SS format?")
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[schema_get_current_datetime]
)

response

Message(id='msg_019snkADUx9cgvrf2LLqnVoR', container=None, content=[ToolUseBlock(id='toolu_01Gm6ht1Si2vzKwPVrFEwpwq', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='tool_get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=720, output_tokens=65, server_tool_use=None, service_tier='standard'))

In [5]:
import json

# Function to support multiple different tools
def run_tool(tool_name, tool_input):
    if tool_name == "tool_get_current_datetime":
        return tool_get_current_datetime(**tool_input)
    elif tool_name == "tool_add_duration_to_datetime":
        return tool_add_duration_to_datetime(**tool_input)
    elif tool_name == "tool_set_reminder":
        return tool_set_reminder(**tool_input)
    # Add more tools as needed

# Function to run tool when there is a tool request
def run_tools(message):
    # There may be more than 1 tool request blocks in the assistant message
    tool_requests = [
        block for block in message.content if block.type == 'tool_use'
    ]

    tool_result_blocks = []
    for tool_request in tool_requests:
        try:
                tool_output = run_tool(tool_request.name, tool_request.input)
                tool_result_block = {
                    "type": "tool_result",
                    "tool_use_id": tool_request.id,
                    "content": json.dumps(tool_output),
                    "is_error": False
                }
        # Error handling
        except Exception as e:
            tool_result_block = {
                    "type": "tool_result",
                    "tool_use_id": tool_request.id,
                    "content": f"Error: {e}",
                    "is_error": True
                }

        tool_result_blocks.append(tool_result_block)
    
    return tool_result_blocks

In [6]:
def run_conversation(messages):
    while True:
        # Call Claude to get assistant message
        response = chat(messages, tools=[schema_get_current_datetime, schema_add_duration_to_datetime, schema_set_reminder])

        # Add assitant message to the list of messages
        add_assistant_message(messages, response)
        print(text_from_message(response))

        # If response from Claude is to ask for tool use, then continue
        if response.stop_reason != 'tool_use':
            break
    
        # Run the tools, get the results
        tool_results = run_tools(response)

        # Add the results to the list of messages, the loop drom the top
        add_user_message(messages, tool_results)

    return messages



In [7]:
# Test out the multiturn tool calling
messages = []
add_user_message(
    messages,
    "Set a reminder for my doctors appointment. It is 100 days from today."
)
run_conversation(messages)

I'll help you set a reminder for your doctor's appointment 100 days from today. Let me first get the current date and then calculate the date that's 100 days from now.
Now let me add 100 days to today's date:
Now I'll set a reminder for your doctor's appointment:
----
Setting the following reminder for 2026-09-06T18:30:43:
Doctor's appointment
----
Perfect! I've set a reminder for your doctor's appointment on **Sunday, September 06, 2026 at 6:30 PM** (100 days from today). You'll receive a notification at that time.


[{'role': 'user',
  'content': 'Set a reminder for my doctors appointment. It is 100 days from today.'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you set a reminder for your doctor's appointment 100 days from today. Let me first get the current date and then calculate the date that's 100 days from now.", type='text'),
   ToolUseBlock(id='toolu_011Yd2AMK7jBqAd2qvwToxpf', caller=DirectCaller(type='direct'), input={'date_format': '%Y-%m-%d %H:%M:%S'}, name='tool_get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_011Yd2AMK7jBqAd2qvwToxpf',
    'content': '"2026-05-29 18:30:43"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Now let me add 100 days to today's date:", type='text'),
   ToolUseBlock(id='toolu_0176smg6orS6xAHXzvDHE8tm', caller=DirectCaller(type='direct'), input={'datetime_str': '2026-05-29 18:30:43', 'duration': 100, 'u